In [20]:
# ============================================
# BUILD PYDECK 3D MAP
# Notebook: 04_map.ipynb  (new notebook)
# ============================================

import json
import pandas as pd
import numpy as np
import pydeck as pdk
import os

# ============================================
# FILE PATHS
# ============================================
geojson_path  = '../data/raw/zaf_admin3.geojson'
master_path   = '../data/cleaned/master_municipal_2021.csv'
cluster_path  = '../data/cleaned/municipal_clusters_2021.csv'
output_path   = '../app/assets/sa_map.html'

# Create app/assets directory if it does not exist
os.makedirs('../app/assets', exist_ok=True)

# ============================================
# LOAD DATA
# ============================================
print("Loading data...")

with open(geojson_path, 'r') as f:
    geojson = json.load(f)

master  = pd.read_csv(master_path)
clusters = pd.read_csv(cluster_path)

print(f"  GeoJSON features: {len(geojson['features'])}")
print(f"  Master dataset:   {len(master)} municipalities")
print(f"  Cluster data:     {len(clusters)} municipalities")

# ============================================
# MERGE CLUSTER AND MASTER DATA
# ============================================
# Merge cluster names into master
master_enriched = master.merge(
    clusters[[
        'muni_code', 'cluster', 'cluster_name',
        'avg_taxable_income', 'governance_score',
        'spending_ratio', 'uifw_pct_budget'
    ]],
    on='muni_code',
    how='left',
    suffixes=('', '_cluster')
)

# ============================================
# CLUSTER COLOUR MAPPING
# RGB arrays for Pydeck
# ============================================
cluster_colors = {
    1: [26, 152, 80, 200],    # Green  — Functional
    0: [252, 141, 89, 200],   # Orange — Dysfunctional
    2: [215, 48, 39, 200],    # Red    — Collapsed
    3: [69, 117, 180, 200],   # Blue   — Crisis
    -1: [150, 150, 150, 150]  # Grey   — No cluster data
}

# ============================================
# ENRICH GEOJSON PROPERTIES
# Inject our data into each feature's properties
# so Pydeck can reference them as
# properties.des_score, properties.cluster etc.
# ============================================
print("\nEnriching GeoJSON with municipal data...")
matched   = 0
unmatched = 0

for feature in geojson['features']:
    props    = feature['properties']
    muni_code = props.get('adm3_name1')

    # Look up this municipality in our master dataset
    row = master_enriched[
        master_enriched['muni_code'] == muni_code
    ]

    if len(row) > 0:
        row = row.iloc[0]
        matched += 1

        # Governance score — drives elevation
        gov_score = row.get('governance_score', 0)
        gov_score = 0 if pd.isna(gov_score) else gov_score

        # Average income — secondary elevation option
        avg_income = row.get('avg_taxable_income', 0)
        avg_income = 0 if pd.isna(avg_income) else avg_income

        # Cluster — drives colour
        cluster = row.get('cluster', -1)
        cluster = -1 if pd.isna(cluster) else int(cluster)

        # UIFW percentage
        uifw = row.get('uifw_pct_budget', 0)
        uifw = 0 if pd.isna(uifw) else uifw

        # Spending ratio
        spend = row.get('spending_ratio', 0)
        spend = 0 if pd.isna(spend) else spend

        # Inject into properties
        props['muni_name']         = str(row.get('muni_name', ''))
        props['province']          = str(row.get('province', ''))
        props['cluster']           = cluster
        props['cluster_name']      = str(row.get('cluster_name', 'No data'))
        props['governance_score']  = round(float(gov_score), 2)
        props['avg_income']        = round(float(avg_income), 0)
        props['uifw_pct']          = round(float(uifw), 1)
        props['spending_ratio']    = round(float(spend), 3)

        # Elevation — governance score scaled up
        # Multiply by 20000 so differences are visible
        # at SA country zoom level
        props['elevation'] = round(float(gov_score) * 20000, 0)

        # Colour array from cluster
        props['fill_color'] = cluster_colors.get(
            cluster, cluster_colors[-1])

    else:
        unmatched += 1
        # Default values for unmatched municipalities
        props['muni_name']        = props.get('adm3_name', '')
        props['province']         = props.get('adm1_name', '')
        props['cluster']          = -1
        props['cluster_name']     = 'No data'
        props['governance_score'] = 0
        props['avg_income']       = 0
        props['uifw_pct']         = 0
        props['spending_ratio']   = 0
        props['elevation']        = 5000
        props['fill_color']       = cluster_colors[-1]

print(f"  Matched:   {matched}")
print(f"  Unmatched: {unmatched}")

# ============================================
# BUILD PYDECK MAP
# ============================================
print("\nBuilding Pydeck map...")

# View state — centred on SA
# pitch=45 gives the 3D perspective
# bearing=0 means north is up
view_state = pdk.ViewState(
    latitude=-28.5,
    longitude=25.5,
    zoom=5,
    pitch=45,
    bearing=0,
    min_zoom=4,
    max_zoom=12
)

# GeoJsonLayer — main 3D municipality layer
geojson_layer = pdk.Layer(
    'GeoJsonLayer',
    geojson,
    opacity=0.85,
    stroked=True,
    filled=True,
    extruded=True,
    wireframe=True,
    pickable=True,
    auto_highlight=True,
    get_elevation='properties.elevation',
    elevation_scale=1,
    get_fill_color='properties.fill_color',
    get_line_color=[255, 255, 255, 30],
    get_line_width=500,
)

# Tooltip — shown on hover
tooltip = {
    'html': '''
        <div style="
            background: rgba(0,0,0,0.8);
            color: white;
            padding: 10px 14px;
            border-radius: 6px;
            font-family: sans-serif;
            font-size: 13px;
            line-height: 1.6;
        ">
            <b style="font-size:15px">
                {properties.muni_name}
            </b><br/>
            <span style="color:#aaa">
                {properties.province}
            </span><br/><br/>
            <b>Cluster:</b>
                {properties.cluster_name}<br/>
            <b>Governance:</b>
                {properties.governance_score} / 5<br/>
            <b>Avg income:</b>
                R{properties.avg_income}<br/>
            <b>UIFW % budget:</b>
                {properties.uifw_pct}%<br/>
            <b>Spending ratio:</b>
                {properties.spending_ratio}
        </div>
    ''',
    'style': {
        'backgroundColor': 'transparent',
        'color': 'white'
    }
}

# Deck object
deck = pdk.Deck(
    layers=[geojson_layer],
    initial_view_state=view_state,
    tooltip=tooltip,
    map_style='mapbox://styles/mapbox/dark-v10',
    parameters={
        'blendColorOperation': 'add',
        'blendColorSrcFactor': 'src-alpha',
        'blendColorDstFactor': 'one-minus-src-alpha',
    }
)

# Export to HTML
deck.to_html(output_path)
print(f"\nMap saved to: {output_path}")
print("Open in browser to view the 3D map.")

# ============================================
# LEGEND SUMMARY
# ============================================
print("\nCluster colour legend:")
print("  Green  (#1a9850) — Cluster 1: Operational — Moderate governance")
print("  Orange (#fc8d59) — Cluster 0: At Risk — High irregular expenditure")
print("  Red    (#d73027) — Cluster 2: Failing — No financial accountability")
print("  Blue   (#4575b4) — Cluster 3: Critical — Extreme budget breach")
print("  Grey             — No cluster data")
print("\nElevation: governance score x 20,000")
print("  Score 5 (clean) = 100,000 unit height")
print("  Score 0 (outstanding) = flat")

Loading data...
  GeoJSON features: 213
  Master dataset:   248 municipalities
  Cluster data:     205 municipalities

Enriching GeoJSON with municipal data...
  Matched:   213
  Unmatched: 0

Building Pydeck map...

Map saved to: ../app/assets/sa_map.html
Open in browser to view the 3D map.

Cluster colour legend:
  Green  (#1a9850) — Cluster 1: Operational — Moderate governance
  Orange (#fc8d59) — Cluster 0: At Risk — High irregular expenditure
  Red    (#d73027) — Cluster 2: Failing — No financial accountability
  Blue   (#4575b4) — Cluster 3: Critical — Extreme budget breach
  Grey             — No cluster data

Elevation: governance score x 20,000
  Score 5 (clean) = 100,000 unit height
  Score 0 (outstanding) = flat


In [21]:
# ============================================
# DIAGNOSE GREY MUNICIPALITIES
# ============================================

import pandas as pd
import json

with open('../data/raw/zaf_admin3.geojson', 'r') as f:
    gj = json.load(f)

master   = pd.read_csv('../data/cleaned/master_municipal_2021.csv')
clusters = pd.read_csv('../data/cleaned/municipal_clusters_2021.csv')

cluster_codes = set(clusters['muni_code'].tolist())
geojson_codes = [
    f['properties']['adm3_name1']
    for f in gj['features']
]

grey = [c for c in geojson_codes if c not in cluster_codes]

print(f"Grey municipalities: {len(grey)}")
print()

for code in sorted(grey):
    row = master[master['muni_code'] == code]
    if len(row) > 0:
        row = row.iloc[0]
        missing = []
        for col in ['avg_taxable_income', 'governance_score',
                    'spending_ratio', 'uifw_pct_budget']:
            if pd.isna(row[col]):
                missing.append(col)
        print(f"{code:<12} {str(row['muni_name']):<35} "
              f"missing: {missing}")
    else:
        print(f"{code:<12} NOT IN MASTER DATASET AT ALL")

Grey municipalities: 8

EC137        Dr. A.B. Xuma                       missing: ['spending_ratio', 'uifw_pct_budget']
KZN224       Impendle                            missing: ['spending_ratio', 'uifw_pct_budget']
KZN261       eDumbe                              missing: ['spending_ratio', 'uifw_pct_budget']
KZN271       Umhlabuyalingana                    missing: ['spending_ratio', 'uifw_pct_budget']
KZN276       Hlabisa Big Five                    missing: ['spending_ratio', 'uifw_pct_budget']
KZN286       Nkandla                             missing: ['spending_ratio', 'uifw_pct_budget']
KZN291       Mandeni                             missing: ['spending_ratio', 'uifw_pct_budget']
NMA          Nelson Mandela Bay                  missing: ['spending_ratio', 'uifw_pct_budget']


## Grey Municipalities — Partial Data Note

### The 8 Municipalities Missing Capex Data

Eight municipalities appeared grey on the initial 3D map
because they were missing capital expenditure budget data
from Municipal Money. Specifically they had no
`spending_ratio` or `uifw_pct_budget` values because
they did not submit capital expenditure returns to
National Treasury for the 2021 financial year.

The 8 affected municipalities:
- Dr. A.B. Xuma (EC137) — Eastern Cape
- Impendle (KZN224) — KwaZulu-Natal
- eDumbe (KZN261) — KwaZulu-Natal
- Umhlabuyalingana (KZN271) — KwaZulu-Natal
- Hlabisa Big Five (KZN276) — KwaZulu-Natal
- Nkandla (KZN286) — KwaZulu-Natal
- Mandeni (KZN291) — KwaZulu-Natal
- Nelson Mandela Bay (NMA) — Eastern Cape

These municipalities were assigned to their nearest
cluster using only income and governance score as
the two available variables. Their cluster label
in the tooltip carries a (partial data) flag to
indicate this.

---

### Nelson Mandela Bay — The Most Significant Case

Nelson Mandela Bay Metropolitan Municipality is one
of SA's 8 metros. It governs Port Elizabeth, Gqeberha,
Uitenhage and Despatch — a combined population of
approximately 1.3 million people.

A full metropolitan municipality failing to submit
capital expenditure data to National Treasury is not
a data quality curiosity. It is a governance signal
of the highest order.

**What it means that NMA was grey**

National Treasury requires all municipalities to
submit Section 71 financial returns quarterly.
Capital expenditure is a core component of these
returns. For a metro of this scale and budget to
have no capex data in the Municipal Money system
for 2021 means one of the following scenarios
occurred.

---

### Possible Scenarios

**Scenario 1 — Submission failure**
NMA submitted returns but the capex component was
incomplete, incorrectly formatted or rejected by
the Treasury system. This is a financial management
failure — a metro with a dedicated CFO and finance
department should be able to submit returns correctly.
Submission failure at this scale suggests either
severe capacity constraints or deliberate avoidance.

**Scenario 2 — Political and administrative instability**
Nelson Mandela Bay has been one of the most politically
unstable metros in SA over the period 2018 to 2023.
Coalition governments collapsed repeatedly. The city
cycled through multiple mayors and city managers in
quick succession. Each change in administration
disrupts financial reporting continuity. A new city
manager inheriting incomplete records from a previous
administration may genuinely be unable to produce
accurate capex returns for periods they did not manage.

**Scenario 3 — Deliberate non-disclosure**
In a forensic accounting context non-submission is
sometimes preferable to submission when the figures
would reveal uncomfortable truths. If capex was
severely underspent — infrastructure projects budgeted
but never started — or severely overspent through
irregular procurement, a non-submission avoids
immediate scrutiny. This is the most concerning
scenario and cannot be ruled out given NMA's broader
governance history during this period.

**Scenario 4 — System migration issues**
The transition to mSCOA (municipal Standard Chart of
Accounts) from 2019 onwards required municipalities
to overhaul their entire financial reporting systems.
Larger municipalities with more complex accounts
sometimes struggled with this transition. NMA may
have had data in their systems that could not be
correctly mapped to the new mSCOA format for
automated submission.

---

### Why This Matters Beyond the Map

NMA managing a multi-billion rand infrastructure
budget with no verifiable capex submission for 2021
means that R-billions in infrastructure spending —
roads, water, electricity, sanitation for 1.3 million
residents — cannot be tracked or verified through
the public data system.

This is the exact scenario the Auditor General flags
when issuing disclaimer opinions. You cannot audit
what has not been submitted. The absence of data is
itself the finding.

For the purposes of this analysis NMA was assigned
to its nearest cluster using income and governance
score only. Its cluster assignment should be treated
as indicative not definitive. A complete financial
picture of NMA requires access to its Annual Financial
Statements directly — not the Municipal Money system.

---

### Data Note for Dashboard Users

Municipalities marked with (partial data) in their
cluster label were assigned using income and governance
score only. Their capex spending and irregular
expenditure figures show as 0 in the tooltip because
no verified data exists in the Municipal Money system
for those variables in the 2021 financial year.

This is not the same as zero irregular expenditure.
Absence of data is not evidence of clean spending.

In [22]:
# ============================================
# FIX GREY MUNICIPALITIES
# Assign partial clusters using income and
# governance score only for the 8 missing capex
# ============================================

import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import json

master   = pd.read_csv('../data/cleaned/master_municipal_2021.csv')
clusters = pd.read_csv('../data/cleaned/municipal_clusters_2021.csv')

# The 8 grey municipality codes
grey_codes = [
    'EC137', 'KZN224', 'KZN261', 'KZN271',
    'KZN276', 'KZN286', 'KZN291', 'NMA'
]

# Get cluster centres from existing clustered data
# Use income and governance only — the two variables
# all 8 grey municipalities have
cluster_centres = clusters.groupby('cluster')[
    ['avg_taxable_income', 'governance_score']
].mean()

print("Cluster centres (income and governance):")
print(cluster_centres.round(0).to_string())
print()

# Scale using same approach as original clustering
scaler_2d = StandardScaler()
centres_scaled = scaler_2d.fit_transform(cluster_centres)

# For each grey municipality find nearest cluster centre
grey_rows = master[master['muni_code'].isin(grey_codes)].copy()

print("Grey municipality assignments:")
assignments = {}
for _, row in grey_rows.iterrows():
    code      = row['muni_code']
    income    = row['avg_taxable_income']
    gov       = row['governance_score']

    if pd.isna(income) or pd.isna(gov):
        # Still cannot assign — leave grey
        assignments[code] = {
            'cluster': -1,
            'cluster_name': 'Insufficient data',
            'partial': True
        }
        print(f"  {code:<12} — still insufficient data")
        continue

    # Scale this municipality's values
    muni_scaled = scaler_2d.transform([[income, gov]])

    # Find nearest cluster centre
    distances = np.linalg.norm(
        centres_scaled - muni_scaled, axis=1)
    nearest_cluster = cluster_centres.index[
        np.argmin(distances)]

    cluster_name_map = {
        0: 'At Risk — High irregular expenditure',
        1: 'Operational — Moderate governance',
        2: 'Failing — No financial accountability',
        3: 'Critical — Extreme budget breach'
    }

    assignments[code] = {
        'cluster':      nearest_cluster,
        'cluster_name': cluster_name_map[nearest_cluster]
                        + ' (partial data)',
        'partial':      True
    }

    print(f"  {code:<12} {str(row['muni_name']):<35} "
          f"→ Cluster {nearest_cluster} "
          f"({cluster_name_map[nearest_cluster]})")

# ============================================
# REBUILD GEOJSON WITH FIXED ASSIGNMENTS
# ============================================
print("\nRebuilding GeoJSON with fixed assignments...")

with open('../data/raw/zaf_admin3.geojson', 'r') as f:
    geojson = json.load(f)

master_enriched = master.merge(
    clusters[[
        'muni_code', 'cluster', 'cluster_name',
        'avg_taxable_income', 'governance_score',
        'spending_ratio', 'uifw_pct_budget'
    ]],
    on='muni_code',
    how='left',
    suffixes=('', '_cluster')
)

cluster_colors = {
    1:  [26, 152, 80, 200],
    0:  [252, 141, 89, 200],
    2:  [215, 48, 39, 200],
    3:  [69, 117, 180, 200],
    -1: [150, 150, 150, 150]
}

matched = unmatched = partial = 0

for feature in geojson['features']:
    props     = feature['properties']
    muni_code = props.get('adm3_name1')

    row = master_enriched[
        master_enriched['muni_code'] == muni_code
    ]

    if len(row) > 0:
        row = row.iloc[0]

        gov_score  = row.get('governance_score', 0)
        gov_score  = 0 if pd.isna(gov_score) else gov_score
        avg_income = row.get('avg_taxable_income', 0)
        avg_income = 0 if pd.isna(avg_income) else avg_income
        uifw       = row.get('uifw_pct_budget', 0)
        uifw       = 0 if pd.isna(uifw) else uifw
        spend      = row.get('spending_ratio', 0)
        spend      = 0 if pd.isna(spend) else spend

        # Check if this is a grey municipality
        # with a partial assignment
        if muni_code in assignments:
            cluster      = assignments[muni_code]['cluster']
            cluster_name = assignments[muni_code]['cluster_name']
            partial += 1
        else:
            cluster      = row.get('cluster', -1)
            cluster_name = row.get('cluster_name', 'No data')
            cluster      = -1 if pd.isna(cluster) \
                           else int(cluster)

        props['muni_name']        = str(row.get('muni_name', ''))
        props['province']         = str(row.get('province', ''))
        props['cluster']          = int(cluster)
        props['cluster_name']     = str(cluster_name)
        props['governance_score'] = round(float(gov_score), 2)
        props['avg_income']       = round(float(avg_income), 0)
        props['uifw_pct']         = round(float(uifw), 1)
        props['spending_ratio']   = round(float(spend), 3)
        props['elevation']        = round(
            float(gov_score) * 20000, 0)
        props['fill_color']       = cluster_colors.get(
            int(cluster), cluster_colors[-1])
        matched += 1

    else:
        unmatched += 1
        props['muni_name']        = props.get('adm3_name', '')
        props['province']         = props.get('adm1_name', '')
        props['cluster']          = -1
        props['cluster_name']     = 'No data'
        props['governance_score'] = 0
        props['avg_income']       = 0
        props['uifw_pct']         = 0
        props['spending_ratio']   = 0
        props['elevation']        = 5000
        props['fill_color']       = cluster_colors[-1]
    
    # UIFW warning flag
    uifw_val = props.get('uifw_pct', 0)
    if uifw_val > 50:
        props['uifw_warning'] = (
            '<div style="margin-top:8px;padding:6px 8px;'
            'background:rgba(215,48,39,0.25);'
            'border-left:3px solid #d73027;'
            'border-radius:3px;font-size:11px;color:#ff9999;">'
            '⚠️ Irregular spend exceeds 50% of budget. '
            'Exercise caution interpreting cluster label.'
            '</div>'
        )
    elif uifw_val > 30:
        props['uifw_warning'] = (
            '<div style="margin-top:8px;padding:6px 8px;'
            'background:rgba(252,141,89,0.25);'
            'border-left:3px solid #fc8d59;'
            'border-radius:3px;font-size:11px;color:#ffcc99;">'
            '⚠️ Irregular spend above 30% of budget.'
            '</div>'
        )
    else:
        props['uifw_warning'] = ''

# Save enriched GeoJSON with warning properties included
enriched_path = '../data/cleaned/sa_municipal_enriched.geojson'
with open(enriched_path, 'w', encoding='utf-8') as f:
   json.dump(geojson, f)
print(f"Enriched GeoJSON saved with WIFW warning.")  

print(f"\n  Matched:   {matched}")
print(f"  Partial:   {partial}")
print(f"  Unmatched: {unmatched}")

# ============================================
# REBUILD AND EXPORT MAP
# ============================================
import pydeck as pdk

view_state = pdk.ViewState(
    latitude=-28.5,
    longitude=25.5,
    zoom=5,
    pitch=45,
    bearing=0,
    min_zoom=4,
    max_zoom=12
)

geojson_layer = pdk.Layer(
    'GeoJsonLayer',
    geojson,
    opacity=0.85,
    stroked=True,
    filled=True,
    extruded=True,
    wireframe=True,
    pickable=True,
    auto_highlight=True,
    get_elevation='properties.elevation',
    elevation_scale=1,
    get_fill_color='properties.fill_color',
    get_line_color=[255, 255, 255, 30],
    get_line_width=500,
)

tooltip = {
    'html': '''
        <div style="
            background: rgba(0,0,0,0.85);
            color: white;
            padding: 12px 16px;
            border-radius: 8px;
            font-family: sans-serif;
            font-size: 13px;
            line-height: 1.8;
            min-width: 240px;
        ">
            <b style="font-size:15px">
                {properties.muni_name}
            </b><br/>
            <span style="color:#aaa;font-size:12px">
                {properties.province}
            </span><br/><br/>
            <b>Cluster:</b>
                {properties.cluster_name}<br/>
            <b>Governance score:</b>
                {properties.governance_score} / 5<br/>
            <b>Avg taxable income:</b>
                R{properties.avg_income}<br/>
            <b>Irregular spend:</b>
                {properties.uifw_pct}% of budget
                <span style="color:#aaa;font-size:11px">
                (unauthorised, irregular or wasteful)
                </span><br/>
            <b>Capex spending ratio:</b>
                {properties.spending_ratio}
                <span style="color:#aaa;font-size:11px">
                (1.0 = spent exactly budget)
                </span>
            {properties.uifw_warning}
        </div>
    ''',
    'style': {
        'backgroundColor': 'transparent',
        'color': 'white'
    }
}

deck = pdk.Deck(
    layers=[geojson_layer],
    initial_view_state=view_state,
    tooltip=tooltip,
    map_style='mapbox://styles/mapbox/dark-v10',
)

deck.to_html('../app/assets/sa_map.html')
print("\nMap rebuilt and saved to app/assets/sa_map.html")
print("Refresh your browser to see the updated map.")

Cluster centres (income and governance):
         avg_taxable_income  governance_score
cluster                                      
0                  257033.0               3.0
1                  270212.0               4.0
2                  239663.0               0.0
3                  247906.0               2.0

Grey municipality assignments:
  EC137        Dr. A.B. Xuma                       → Cluster 2 (Failing — No financial accountability)
  KZN224       Impendle                            → Cluster 3 (Critical — Extreme budget breach)
  KZN261       eDumbe                              → Cluster 3 (Critical — Extreme budget breach)
  KZN271       Umhlabuyalingana                    → Cluster 3 (Critical — Extreme budget breach)
  KZN276       Hlabisa Big Five                    → Cluster 3 (Critical — Extreme budget breach)
  KZN286       Nkandla                             → Cluster 1 (Operational — Moderate governance)
  KZN291       Mandeni                             → Clus

c:\Users\banel\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\banel\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\banel\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\banel\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\banel\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2691: U

Enriched GeoJSON saved with WIFW warning.

  Matched:   213
  Partial:   8
  Unmatched: 0

Map rebuilt and saved to app/assets/sa_map.html
Refresh your browser to see the updated map.


In [23]:
# ============================================
# REBUILD MAP — FLAT WITH COLOURS
# ============================================

import json
import pydeck as pdk

# Load enriched GeoJSON from saved file
enriched_path = '../data/cleaned/sa_municipal_enriched.geojson'

with open(enriched_path, 'r', encoding='utf-8') as f:
    geojson_enriched = json.load(f)

print("Enriched GeoJSON loaded.")
sample = geojson_enriched['features'][0]['properties']
print(f"  Sample muni: {sample.get('muni_name')}")
print(f"  Cluster:     {sample.get('cluster_name')}")
print(f"  Fill color:  {sample.get('fill_color')}")

view_state = pdk.ViewState(
    latitude=-28.5,
    longitude=25.5,
    zoom=5,
    pitch=0,
    bearing=0,
    min_zoom=4,
    max_zoom=12
)

geojson_layer = pdk.Layer(
    'GeoJsonLayer',
    geojson_enriched,
    opacity=0.85,
    stroked=True,
    filled=True,
    extruded=False,
    pickable=True,
    auto_highlight=True,
    get_fill_color='properties.fill_color',
    get_line_color=[255, 255, 255, 60],
    get_line_width=500,
)

tooltip = {
    'html': '''
        <div style="
            background: rgba(0,0,0,0.85);
            color: white;
            padding: 12px 16px;
            border-radius: 8px;
            font-family: sans-serif;
            font-size: 13px;
            line-height: 1.8;
            min-width: 240px;
        ">
            <b style="font-size:15px">
                {properties.muni_name}
            </b><br/>
            <span style="color:#aaa;font-size:12px">
                {properties.province}
            </span><br/><br/>
            <b>Cluster:</b>
                {properties.cluster_name}<br/>
            <b>Governance score:</b>
                {properties.governance_score} / 5<br/>
            <b>Avg taxable income:</b>
                R{properties.avg_income}<br/>
            <b>Irregular spend:</b>
                {properties.uifw_pct}% of budget
                <span style="color:#aaa;font-size:11px">
                (unauthorised, irregular or wasteful)
                </span><br/>
            <b>Capex spending ratio:</b>
                {properties.spending_ratio}
                <span style="color:#aaa;font-size:11px">
                (1.0 = spent exactly budget)
                </span>
            {properties.uifw_warning}
        </div>
    ''',
    'style': {
        'backgroundColor': 'transparent',
        'color': 'white'
    }
}

deck = pdk.Deck(
    layers=[geojson_layer],
    initial_view_state=view_state,
    tooltip=tooltip,
    map_style='https://basemaps.cartocdn.com/gl/dark-matter-nolabels-gl-style/style.json',
)

output_path = '../app/assets/sa_map.html'
deck.to_html(output_path)

# Inject legend
legend_html = '''
<div style="
    position: fixed;
    bottom: 30px;
    left: 30px;
    background: rgba(0,0,0,0.82);
    color: white;
    padding: 16px 20px;
    border-radius: 10px;
    font-family: sans-serif;
    font-size: 13px;
    line-height: 2;
    z-index: 1000;
    min-width: 260px;
    box-shadow: 0 4px 12px rgba(0,0,0,0.4);
">
    <div style="font-size:15px;font-weight:bold;
                margin-bottom:10px;border-bottom:
                1px solid rgba(255,255,255,0.2);
                padding-bottom:8px;">
        SA Municipality Clusters — 2021
    </div>
    <div>
        <span style="display:inline-block;width:14px;
                     height:14px;border-radius:3px;
                     background:#1a9850;
                     margin-right:10px;
                     vertical-align:middle;">
        </span>
        Operational — Moderate governance (n=125)
    </div>
    <div>
        <span style="display:inline-block;width:14px;
                     height:14px;border-radius:3px;
                     background:#fc8d59;
                     margin-right:10px;
                     vertical-align:middle;">
        </span>
        At Risk — High irregular expenditure (n=50)
    </div>
    <div>
        <span style="display:inline-block;width:14px;
                     height:14px;border-radius:3px;
                     background:#d73027;
                     margin-right:10px;
                     vertical-align:middle;">
        </span>
        Failing — No financial accountability (n=28)
    </div>
    <div>
        <span style="display:inline-block;width:14px;
                     height:14px;border-radius:3px;
                     background:#4575b4;
                     margin-right:10px;
                     vertical-align:middle;">
        </span>
        Critical — Extreme budget breach (n=2)
    </div>
    <div style="margin-top:10px;padding-top:8px;
                border-top:1px solid rgba(255,255,255,0.2);
                color:#aaa;font-size:11px;">
        Hover over municipality for full profile<br/>
        Source: National Treasury, SARS — 2021
    </div>
</div>
'''

with open(output_path, 'r', encoding='utf-8') as f:
    html_content = f.read()

html_content = html_content.replace(
    '</body>',
    legend_html + '\n</body>'
)

with open(output_path, 'w', encoding='utf-8') as f:
    f.write(html_content)

print("Flat map saved. Hard refresh browser (Ctrl+Shift+R).")

Enriched GeoJSON loaded.
  Sample muni: Abaqulusi
  Cluster:     At Risk — High irregular expenditure
  Fill color:  [252, 141, 89, 200]
Flat map saved. Hard refresh browser (Ctrl+Shift+R).


In [24]:
# ============================================
# REBUILD HTML FROM UPDATED GEOJSON
# ============================================
import json
import pydeck as pdk

# Load the already-fixed enriched GeoJSON
enriched_path = '../data/cleaned/sa_municipal_enriched.geojson'

with open(enriched_path, 'r', encoding='utf-8') as f:
    geojson_enriched = json.load(f)

# Verify names are correct before building
sample_names = set(
    f['properties']['cluster_name']
    for f in geojson_enriched['features']
)
print("Cluster names in GeoJSON:")
for name in sorted(sample_names):
    print(f"  {name}")

view_state = pdk.ViewState(
    latitude=-28.5,
    longitude=25.5,
    zoom=5,
    pitch=0,
    bearing=0,
    min_zoom=4,
    max_zoom=12
)

geojson_layer = pdk.Layer(
    'GeoJsonLayer',
    geojson_enriched,
    opacity=0.85,
    stroked=True,
    filled=True,
    extruded=False,
    pickable=True,
    auto_highlight=True,
    get_fill_color='properties.fill_color',
    get_line_color=[255, 255, 255, 60],
    get_line_width=500,
)

tooltip = {
    'html': '''
        <div style="
            background: rgba(0,0,0,0.85);
            color: white;
            padding: 12px 16px;
            border-radius: 8px;
            font-family: sans-serif;
            font-size: 13px;
            line-height: 1.8;
            min-width: 240px;
        ">
            <b style="font-size:15px">
                {properties.muni_name}
            </b><br/>
            <span style="color:#aaa;font-size:12px">
                {properties.province}
            </span><br/><br/>
            <b>Cluster:</b>
                {properties.cluster_name}<br/>
            <b>Governance score:</b>
                {properties.governance_score} / 5<br/>
            <b>Avg taxable income:</b>
                R{properties.avg_income}<br/>
            <b>Irregular spend:</b>
                {properties.uifw_pct}% of budget
                <span style="color:#aaa;font-size:11px">
                (unauthorised, irregular or wasteful)
                </span><br/>
            <b>Capex spending ratio:</b>
                {properties.spending_ratio}
                <span style="color:#aaa;font-size:11px">
                (1.0 = spent exactly budget)
                </span>
            {properties.uifw_warning}
        </div>
    ''',
    'style': {
        'backgroundColor': 'transparent',
        'color': 'white'
    }
}

deck = pdk.Deck(
    layers=[geojson_layer],
    initial_view_state=view_state,
    tooltip=tooltip,
    map_style='https://basemaps.cartocdn.com/gl/dark-matter-nolabels-gl-style/style.json',
)

output_path = '../app/assets/sa_map.html'
deck.to_html(output_path)

# Inject legend with updated names
legend_html = '''
<div style="
    position: fixed;
    bottom: 30px;
    left: 30px;
    background: rgba(0,0,0,0.82);
    color: white;
    padding: 16px 20px;
    border-radius: 10px;
    font-family: sans-serif;
    font-size: 13px;
    line-height: 2;
    z-index: 1000;
    min-width: 260px;
    box-shadow: 0 4px 12px rgba(0,0,0,0.4);
">
    <div style="font-size:15px;font-weight:bold;
                margin-bottom:10px;border-bottom:
                1px solid rgba(255,255,255,0.2);
                padding-bottom:8px;">
        SA Municipality Clusters — 2021
    </div>
    <div>
        <span style="display:inline-block;width:14px;
                     height:14px;border-radius:3px;
                     background:#1a9850;
                     margin-right:10px;
                     vertical-align:middle;">
        </span>
        Operational — Moderate governance (n=125)
    </div>
    <div>
        <span style="display:inline-block;width:14px;
                     height:14px;border-radius:3px;
                     background:#fc8d59;
                     margin-right:10px;
                     vertical-align:middle;">
        </span>
        At Risk — High irregular expenditure (n=50)
    </div>
    <div>
        <span style="display:inline-block;width:14px;
                     height:14px;border-radius:3px;
                     background:#d73027;
                     margin-right:10px;
                     vertical-align:middle;">
        </span>
        Failing — No financial accountability (n=28)
    </div>
    <div>
        <span style="display:inline-block;width:14px;
                     height:14px;border-radius:3px;
                     background:#4575b4;
                     margin-right:10px;
                     vertical-align:middle;">
        </span>
        Critical — Extreme budget breach (n=2)
    </div>
    <div style="margin-top:10px;padding-top:8px;
                border-top:1px solid rgba(255,255,255,0.2);
                color:#aaa;font-size:11px;">
        Hover over municipality for full profile<br/>
        Clusters based on 2021 financial data only.<br/>
        Relative positioning — not absolute quality.<br/>
        Source: National Treasury, SARS — 2021
    </div>
</div>
'''

with open(output_path, 'r', encoding='utf-8') as f:
    html_content = f.read()

html_content = html_content.replace(
    '</body>',
    legend_html + '\n</body>'
)

with open(output_path, 'w', encoding='utf-8') as f:
    f.write(html_content)

print("\nMap rebuilt from updated GeoJSON.")
print("Hard refresh browser (Ctrl+Shift+R).")

Cluster names in GeoJSON:
  At Risk — High irregular expenditure
  Critical — Extreme budget breach
  Critical — Extreme budget breach (partial data)
  Failing — No financial accountability
  Failing — No financial accountability (partial data)
  Operational — Moderate governance
  Operational — Moderate governance (partial data)

Map rebuilt from updated GeoJSON.
Hard refresh browser (Ctrl+Shift+R).
